In [1]:
import pandas as pd
import pyterrier as pt
if not pt.started():
    pt.init()

import sys
sys.path.append("./lsr_package")
from lsr.transformer import LSR
from pyterrier_pisa import PisaIndex
from ir_measures import *

/tmp/ipykernel_11299/4160717908.py:3: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():
Java started and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/tmp/ipykernel_11299/4160717908.py:4: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()
/opt/miniconda3/envs/splade_class/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
docs = pd.read_pickle("/nfs/primary/sas_reranker/ohsumed_docs_w_t5base_sensitivity.pkl")
queries = pd.read_pickle("/nfs/primary/graph_adaptive_reranking/queries.pkl")
qrels = pd.read_pickle("/nfs/primary/graph_adaptive_reranking/qrels.pkl")

# Util functions
def ohsumed_iterator():
    for row in tqdm(docs.iterrows(), total = len(docs), desc = "Indexing"):
        row = row[1]
        yield {"docno" : row.docno, "text" : row.text}

def _sens_docs(qrels, run):
    if "sensitivity" in run.columns:
        run = run.drop(columns = ["sensitivity"])
    merged = pd.merge(run, docs, left_on = "doc_id", right_on = "docno")
    return merged.sensitivity.sum()
    
import ir_measures
sens_docs = ir_measures.define_byquery(
    _sens_docs, 
    name="sens_docs")

In [3]:
model_folder = "/nfs/primary/SPLADE/splade_class/lsr_package/outputs/splade_ohsumed_multiple_negative/model/"
index_folder = "/nfs/primary/SPLADE/splade_class/indices/sens_reg_exp/splade_ohsumed_multiple_negative"
lsr = LSR(model_folder)
index = PisaIndex(index_folder)

In [4]:
splade_retr = lsr.query_encoder() >> index.quantized()
run_df = splade_retr(queries)

In [5]:
pt.io.write_results(run_df, "./runs/splade_ohsumed_multiple_negative")

In [6]:
def filter(run):
    docs_w_svm_labels = pd.read_csv("./ohsumed_w_svm_sensitivity.csv")
    docs_w_svm_labels.docno = docs_w_svm_labels.docno.astype(str)
    new_run = pd.merge(run, docs_w_svm_labels, on = "docno")
    assert len(run) == len(new_run)
    return new_run[new_run["svm_labels"] == 0]

splade_filter_retr = lsr.query_encoder() >> index.quantized() >> pt.apply.generic(filter)
svm_filter_run_df = splade_filter_retr(queries)

In [7]:
pt.io.write_results(svm_filter_run_df, "./runs/splade_ohsumed_multiple_negative_svm_filter")

In [8]:
retrieval_results = pt.Experiment(
    [
        pt.io.read_results("./runs/splade_ohsumed_multiple_negative"),
        pt.io.read_results("./runs/splade_ohsumed_multiple_negative_svm_filter"),
        pt.io.read_results("./runs/splade_ohsumed_multiple_negative_t5_filter")
    ],
    queries,
    qrels,
    eval_metrics=[nDCG@10, sens_docs@10],
    names = [
        "SPLADE",
        "SPLADE >> SVM Filter",
        "SPLADE >> T5 Filter"
    ]
)

retrieval_results

,name,nDCG@10,sens_docs@10
0,SPLADE,0.437492,1.216981
1,SPLADE >> SVM Filter,0.360254,0.943396
2,SPLADE >> T5 Filter,0.428907,1.028302
